# AfriMeet AI — Whisper Fine-Tuning (Colab GPU)

Hybrid workflow: data download/prep can run locally or here; this notebook does the
GPU-heavy parts — Phase 3 (baseline evaluation), Phase 4 (fine-tuning), and Phase 5
(comparison) — using Colab's free GPU. It doesn't duplicate any pipeline logic: every
cell below just calls into the same `afrimeet` package and `scripts/` used locally.
Google Drive is used to persist data and checkpoints between sessions, since Colab's
local disk is wiped when the runtime disconnects.

**Before you start:**
1. `Runtime -> Change runtime type -> GPU` (T4 is fine).
2. The code is cloned straight from the public GitHub repo (`REPO_URL` below) — no
   authentication needed.
   - If you'd rather not clone from GitHub, clear `REPO_URL` to fall back to the
     Drive-zip method instead: run `python scripts/package_for_colab.py` locally and
     upload `dist/afrimeet-ai-code.zip` to `My Drive/AfriMeet_AI/afrimeet-ai-code.zip`.

Both configured datasets (Common Voice via the `fsicoli/common_voice_15_0` community
mirror — see the note in `configs/config.yaml` for why not the official one — and
FLEURS) are ungated, so no Hugging Face login is required to run this notebook
end to end.

Phase 6 (the API / web app) isn't part of this notebook — it's meant to run wherever
you deploy it, using the fine-tuned model this notebook produces.

In [ ]:
# Confirm a GPU is attached to this runtime (Runtime -> Change runtime type -> GPU
# if this errors or shows no devices).
!nvidia-smi

In [ ]:
# Mount Google Drive — this is where the processed dataset, model checkpoints, and
# evaluation reports get backed up so they survive between Colab sessions.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Config: where the code lives locally on this VM (WORK_DIR), where persistent
# storage lives on Drive (DRIVE_ROOT), and where to get the code from (REPO_URL).
import os

PROJECT_NAME = "afrimeet-ai"
WORK_DIR = f"/content/{PROJECT_NAME}"
DRIVE_ROOT = "/content/drive/MyDrive/AfriMeet_AI"

# Public repo -> plain clone, no auth needed. Clear this (set to "") to fall back to
# the Drive-zip method instead.
REPO_URL = "https://github.com/claverfred/afrimeet-ai.git"
CODE_ZIP_PATH = f"{DRIVE_ROOT}/afrimeet-ai-code.zip"

os.makedirs(DRIVE_ROOT, exist_ok=True)
print("WORK_DIR:", WORK_DIR)
print("DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
# Get the project code onto this VM: clone from GitHub (default), or unpack a
# Drive-uploaded zip if REPO_URL was cleared above.
import shutil
import subprocess

if REPO_URL:
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    assert os.path.exists(CODE_ZIP_PATH), (
        f"{CODE_ZIP_PATH} not found. Run `python scripts/package_for_colab.py` "
        "locally and upload the resulting zip to that Drive path, or set REPO_URL above."
    )
    os.makedirs(WORK_DIR, exist_ok=True)
    shutil.unpack_archive(CODE_ZIP_PATH, WORK_DIR)
    print(f"Unpacked {CODE_ZIP_PATH} -> {WORK_DIR}")

In [ ]:
# cd into the cloned/unpacked repo and install all dependencies (Colab's default
# environment doesn't have these — this is separate from your local .venv).
%cd $WORK_DIR
!pip install -q -r requirements/base.txt -r requirements/ml.txt -r requirements/api.txt
!pip install -q -e . --no-deps

# The editable install above writes a .pth file that only gets picked up by a *new*
# Python process (which is why the `!python scripts/...` calls later in this notebook
# work fine) — this kernel is already running, so `import afrimeet` won't see it until
# we add src/ to sys.path directly.
import sys
sys.path.insert(0, f"{WORK_DIR}/src")

In [ ]:
# Sanity check: confirm PyTorch sees the GPU and the afrimeet package imports
# correctly before running any real work below.
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

import afrimeet
print("afrimeet package loaded, version", afrimeet.__version__)

In [ ]:
# Optional: authenticate with Hugging Face. Both configured datasets (Common Voice
# via fsicoli/common_voice_15_0, and FLEURS) are ungated, so this isn't required —
# only useful if you swap in a gated dataset later, or want higher download rate
# limits. Leave blank to skip.
from getpass import getpass

from huggingface_hub import login

hf_token = getpass("Hugging Face token (blank to skip): ")
if hf_token:
    login(token=hf_token)
del hf_token

## Phase 2 — Data

Trains on Common Voice, evaluates on FLEURS's test split only (never trained on) —
see the comment block in `configs/config.yaml` for why. Restores the processed
dataset from a Drive-cached zip if one exists (fast, no re-download). Otherwise
downloads + prepares it fresh on the local Colab disk (fast I/O — writing thousands
of small audio files directly to a Drive-mounted path is much slower) and caches the
result back to Drive for next time.

In [ ]:
# Phase 2 — Data: restore the processed dataset from a Drive-cached zip if one
# exists; otherwise download + prepare it fresh (calls scripts/download_data.py and
# scripts/prepare_dataset.py, the exact same scripts used locally) and cache the
# result to Drive for next time.
#
# Uses subprocess.run(check=True) rather than `!python ...` so a failure here raises
# immediately with a clear traceback, instead of silently leaving an empty dataset
# that only surfaces as a confusing error several cells later. The restored-from-cache
# path is also validated and self-heals (discards + rebuilds) if the cached zip is
# stale — e.g. left over from before a configs/config.yaml dataset change.
import shutil
import subprocess
from pathlib import Path

data_zip = Path(DRIVE_ROOT) / "data_processed.zip"
processed_dir = Path(WORK_DIR) / "data" / "processed"


def _has_required_manifests(d: Path) -> bool:
    return any(d.glob("*/train/manifest.csv")) and any(d.glob("*/test/manifest.csv"))


restored_ok = False
if data_zip.exists():
    print(f"Restoring processed dataset from {data_zip} ...")
    processed_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(data_zip), str(processed_dir))
    restored_ok = _has_required_manifests(processed_dir)
    if not restored_ok:
        print(
            f"{data_zip} is missing a train and/or test manifest — likely stale from "
            "before a configs/config.yaml dataset change. Discarding it and rebuilding fresh."
        )

if not restored_ok:
    if processed_dir.exists():
        shutil.rmtree(processed_dir)
    print("Downloading and preparing the dataset from scratch (this can take a while) ...")
    subprocess.run(["python", "scripts/download_data.py"], check=True)
    subprocess.run(["python", "scripts/prepare_dataset.py"], check=True)

    if not _has_required_manifests(processed_dir):
        raise RuntimeError(
            f"{processed_dir} is still missing a train and/or test manifest after a "
            "fresh download. Scroll up through download_data.py/prepare_dataset.py "
            "output for warnings (e.g. a dataset that failed to download, or produced "
            "zero usable examples after cleaning)."
        )

    print(f"Caching processed dataset to {data_zip} for future sessions ...")
    shutil.make_archive(str(data_zip.with_suffix("")), "zip", root_dir=str(processed_dir))

## Phase 3 — Baseline evaluation (pre-trained Whisper)

In [ ]:
# Locate a processed test-split manifest to evaluate both models against
# (used by both the Phase 3 baseline eval and the Phase 5 fine-tuned eval below).
import glob

test_manifests = glob.glob(f"{WORK_DIR}/data/processed/*/test/manifest.csv")
assert test_manifests, "No test manifest found under data/processed/*/test/manifest.csv"
TEST_MANIFEST = test_manifests[0]
print("Using test manifest:", TEST_MANIFEST)

In [ ]:
# Run the pre-trained ("baseline") Whisper model through evaluate_model.py -> writes
# reports/metrics/baseline_summary.json and baseline_per_example.csv (WER/CER/latency).
# subprocess.run(check=True) so a failure here stops the notebook with a clear
# traceback instead of silently letting later cells run on missing/stale results.
import subprocess

subprocess.run(
    [
        "python",
        "scripts/evaluate_model.py",
        "--manifest",
        TEST_MANIFEST,
        "--model",
        "openai/whisper-small",
        "--run-name",
        "baseline",
    ],
    check=True,
)

## Phase 4 — Fine-tune Whisper on the conference-domain data

Hyperparameters come from `configs/config.yaml` (`training:` section). Lower
`train_batch_size` there if you hit a CUDA out-of-memory error on the T4's 16GB.

If a **completed** fine-tuned model backup already exists on Drive from a previous
session, this restores it instead of retraining — set `RETRAIN = True` to force a
fresh run. Separately, `train.py` itself backs up its *in-progress* checkpoint to
Drive periodically (every `save_steps`) and auto-resumes from it if found — so a long
overnight run that gets disconnected can pick back up close to where it left off
instead of losing everything, as long as at least one checkpoint was saved before the
disconnect.

In [ ]:
# Phase 4 setup: if a fine-tuned model was already backed up to Drive in a previous
# session, restore it and skip retraining. Set RETRAIN = True to force a fresh run
# (e.g. after changing hyperparameters in configs/config.yaml).
#
# Also points train.py at a Drive folder for mid-training checkpoint backups (read via
# the AFRIMEET_CHECKPOINT_BACKUP_DIR env var, which `!python scripts/train.py` inherits
# from this kernel) — this is what lets a disconnected overnight run resume instead of
# starting over. See afrimeet/models/train.py's CheckpointBackupCallback.
import os
import shutil
from pathlib import Path

finetuned_dir = Path(WORK_DIR) / "models" / "finetuned"
backup_zip = Path(DRIVE_ROOT) / "models_finetuned.zip"

os.environ["AFRIMEET_CHECKPOINT_BACKUP_DIR"] = f"{DRIVE_ROOT}/checkpoint_backup"

RETRAIN = False

if backup_zip.exists() and not RETRAIN:
    print(f"Found existing backup at {backup_zip} — restoring instead of retraining.")
    finetuned_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(backup_zip), str(finetuned_dir))
    SKIP_TRAINING = True
else:
    SKIP_TRAINING = False

In [ ]:
# Fine-tune Whisper via train.py (uses configs/config.yaml's training: hyperparameters),
# unless a restored model made this unnecessary (see cell above).
#
# subprocess.run(check=True) rather than `!python ...` so a training failure stops this
# notebook immediately with a clear traceback, instead of silently falling through to
# the backup cell below and zipping up whatever partial/empty output was left behind
# (this is what produced the tiny, broken models_finetuned.zip from an earlier run).
import subprocess

if not SKIP_TRAINING:
    subprocess.run(["python", "scripts/train.py"], check=True)
else:
    print("Skipping training — using the model restored from Drive. Set RETRAIN = True above to force retraining.")

In [ ]:
# Back up the fine-tuned model to Drive so it survives past this session (safe to
# re-run any time, including mid-training from a second cell if you're worried about a
# disconnect on a long run).
#
# Validates a real weight file exists first -- a crashed/incomplete run leaves only
# small config/tokenizer files behind, and backing that up as if it were a real model
# would silently corrupt Phase 5's evaluation later with no obvious error.
import shutil
from pathlib import Path

weight_files = [
    f
    for f in list(finetuned_dir.rglob("model.safetensors")) + list(finetuned_dir.rglob("pytorch_model.bin"))
    if f.stat().st_size > 10_000_000  # >10MB -- filters out empty/truncated saves
]
if not weight_files:
    raise RuntimeError(
        f"No real model weight file (>10MB) found under {finetuned_dir} -- training "
        "likely failed or didn't complete. Refusing to back this up. Check the "
        "training cell's output/traceback and re-run it."
    )

backup_path = shutil.make_archive(str(backup_zip.with_suffix("")), "zip", root_dir=str(finetuned_dir))
weights_mb = weight_files[0].stat().st_size / 1e6
print(f"Backed up fine-tuned model to {backup_path} (found {weights_mb:.0f}MB weights file)")

## Phase 5 — Evaluate the fine-tuned model and compare against baseline

In [ ]:
# Resolve the fine-tuned model's path from configs/config.yaml (same config the
# training cell used to decide where to save it).
from afrimeet.utils.config import load_config

config = load_config()
FINETUNED_MODEL = f"{config['paths']['models_finetuned']}/{config['whisper']['finetuned_model_name']}"
print("Using fine-tuned model:", FINETUNED_MODEL)

In [ ]:
# Run the fine-tuned model through the same evaluate_model.py used for the baseline ->
# writes reports/metrics/finetuned_summary.json and finetuned_per_example.csv.
import subprocess

subprocess.run(
    [
        "python",
        "scripts/evaluate_model.py",
        "--manifest",
        TEST_MANIFEST,
        "--model",
        FINETUNED_MODEL,
        "--run-name",
        "finetuned",
    ],
    check=True,
)

In [ ]:
# Compare the baseline vs. fine-tuned summaries side by side (WER/CER relative
# improvement) -> writes reports/metrics/comparison.csv.
import subprocess

subprocess.run(
    [
        "python",
        "scripts/compare_models.py",
        "--runs",
        "baseline=reports/metrics/baseline_summary.json",
        "finetuned=reports/metrics/finetuned_summary.json",
    ],
    check=True,
)

In [ ]:
# Back up all evaluation reports (baseline + fine-tuned summaries, comparison table)
# to Drive so they're not lost when this session ends.
import shutil
from pathlib import Path

reports_src = Path(WORK_DIR) / "reports" / "metrics"
reports_backup = Path(DRIVE_ROOT) / "reports_metrics"
if reports_backup.exists():
    shutil.rmtree(reports_backup)
shutil.copytree(reports_src, reports_backup)
print(f"Backed up metrics to {reports_backup}")

## Resuming in a later session

Just re-run the cells from the top:
- The data cell finds `data_processed.zip` on Drive and skips re-downloading.
- The Phase 4 setup cell finds `models_finetuned.zip` on Drive (a **completed** run)
  and skips retraining entirely, restoring that model. Set `RETRAIN = True` to
  fine-tune again — e.g. after changing hyperparameters in `configs/config.yaml`.
- If training was interrupted partway through instead (e.g. an overnight disconnect),
  there's no `models_finetuned.zip` yet, but `train.py` finds the mid-run checkpoint
  backup at `{DRIVE_ROOT}/checkpoint_backup/latest_checkpoint` (written every
  `save_steps`) and resumes from there automatically instead of starting over. Worst
  case you lose progress since the last checkpoint, not the whole run.